# Imports

In [1]:
import geopandas as gpd

## Read and Inspect the Data


In [2]:
import xarray as xr
# import xclim.indices

# Set xarray to use HTML for displaying outputs
from pathlib import Path
nc_file_path = Path("../cama_flood/out/my_test_simulation") / "o_maxdph1980.nc"
# Open the NetCDF file
ds = xr.open_dataset(nc_file_path)

# Display dataset information
print("Dataset information:")

print("\n" + "="*50)
print("Variable information:")
print(ds.data_vars)

print("\n" + "="*50)
print("Coordinates:")
print(ds.coords)


Dataset information:

Variable information:
Data variables:
    maxdph   (lon, lat, time) float32 25MB ...

Coordinates:
Coordinates:
  * lon      (lon) float64 12kB -180.0 -179.7 -179.5 ... 179.5 179.7 180.0
  * lat      (lat) float64 6kB 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * time     (time) datetime64[ns] 48B 1980-01-01 1980-01-02 ... 1980-01-06


In [3]:
ds

<xarray.Dataset> Size: 25MB
Dimensions:  (lon: 1440, lat: 720, time: 6)
Coordinates:
  * lon      (lon) float64 12kB -180.0 -179.7 -179.5 ... 179.5 179.7 180.0
  * lat      (lat) float64 6kB 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * time     (time) datetime64[ns] 48B 1980-01-01 1980-01-02 ... 1980-01-06
Data variables:
    maxdph   (lon, lat, time) float32 25MB ...

In [4]:
maxdph = ds["maxdph"]  # or whatever the variable name is

# # 2. (optional) require at least 360 valid days per year
# valid_counts = runoff.groupby("time.year").count(dim="time")
# mask = valid_counts > 360

# Annual max runoff
yearly_max = maxdph.groupby("time.year").max(dim="time") #.where(mask)


In [5]:
yearly_max

<xarray.DataArray 'maxdph' (lon: 1440, lat: 720, year: 1)> Size: 4MB
array([[[1.e+20],
        [1.e+20],
        [1.e+20],
        ...,
        [1.e+20],
        [1.e+20],
        [1.e+20]],

       [[1.e+20],
        [1.e+20],
        [1.e+20],
        ...,
        [1.e+20],
        [1.e+20],
        [1.e+20]],

       [[1.e+20],
        [1.e+20],
        [1.e+20],
        ...,
...
        ...,
        [1.e+20],
        [1.e+20],
        [1.e+20]],

       [[1.e+20],
        [1.e+20],
        [1.e+20],
        ...,
        [1.e+20],
        [1.e+20],
        [1.e+20]],

       [[1.e+20],
        [1.e+20],
        [1.e+20],
        ...,
        [1.e+20],
        [1.e+20],
        [1.e+20]]], shape=(1440, 720, 1), dtype=float32)
Coordinates:
  * lon      (lon) float64 12kB -180.0 -179.7 -179.5 ... 179.5 179.7 180.0
  * lat      (lat) float64 6kB 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * year     (year) int64 8B 1980
Attributes:
    long_name:  Maximum depth
    units:      m

In [6]:
import os
store = "../data/yearly_max_runoff.zarr"
# mode = "a" if os.path.exists(store) else "w"
mode = "w"
# In append mode, data for an existing year will be duplicated
yearly_max.to_zarr(store, mode=mode) #, append_dim="year")

/home/hamada/python_projects/flood_risks/.pixi/envs/default/lib/python3.11/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [7]:
yearly_max = xr.open_zarr(store)
dph = yearly_max["maxdph"].to_dataframe().reset_index()
dph = gpd.GeoDataFrame(dph, geometry=gpd.points_from_xy(dph["lon"], dph["lat"]), crs="EPSG:4326")

In [8]:
dph

,lon,lat,year,maxdph,geometry
0,-180.0,90.000000,1980,1.000000e+20,POINT (-180 90)
1,-180.0,89.749652,1980,1.000000e+20,POINT (-180 89.74965)
2,-180.0,89.499305,1980,1.000000e+20,POINT (-180 89.4993)
3,-180.0,89.248957,1980,1.000000e+20,POINT (-180 89.24896)
4,-180.0,88.998609,1980,1.000000e+20,POINT (-180 88.99861)
...,...,...,...,...,...
1036795,180.0,-88.998609,1980,1.000000e+20,POINT (180 -88.99861)
1036796,180.0,-89.248957,1980,1.000000e+20,POINT (180 -89.24896)
1036797,180.0,-89.499305,1980,1.000000e+20,POINT (180 -89.4993)
1036798,180.0,-89.749652,1980,1.000000e+20,POINT (180 -89.74965)


In [9]:
import zipfile
import pandas as pd
from fnmatch import fnmatch
from pathlib import Path

usecols = ["source_id", "lat", "lon", "capacity"]
zip_path = Path("../data/power.zip")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    
    # Pattern to match (works in any subfolder)
    pattern = "*_emissions_sources_v*.csv"
    matching_files = [f for f in all_files if fnmatch(f.split('/')[-1], pattern)]
    
    if matching_files:
        csv_path = matching_files[0]
        print(f"Reading: {csv_path}")
        
        with zip_ref.open(csv_path) as csv_file:
            power = pd.read_csv(csv_file, usecols=usecols)
            print(power.head())
    else:
        print(f"No files found matching pattern: {pattern}")

Reading: DATA/electricity-generation_emissions_sources_v4_8_0.csv
   source_id      lat      lon  capacity
0   25452484  12.4759 -69.9823     254.5
1   25452484  12.4759 -69.9823     254.5
2   25452484  12.4759 -69.9823     254.5
3   25452484  12.4759 -69.9823     254.5
4   25452484  12.4759 -69.9823     254.5


In [10]:
power = gpd.GeoDataFrame(power, geometry=gpd.points_from_xy(power["lon"], power["lat"]), crs="EPSG:4326")

In [14]:
merged = gpd.sjoin_nearest(power.to_crs("EPSG:3857"), dph.to_crs("EPSG:3857"), max_distance=50_000, distance_col="dist")

In [17]:
merged.to_parquet("../data/exposures/power.parquet")

In [ ]:
# next step: format and merge with damage function